In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier



In [3]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
X = df.drop(columns=["stroke", "id"])
y = df["stroke"]

In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

In [9]:
categorical_features = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

numerical_features = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi"
]

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

In [19]:
for n in [50, 100, 200, 300]:

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=n,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    val_score = model.score(X_val, y_val)

    print(f"n_estimators = {n}, Validation Score = {val_score:.4f}")

n_estimators = 50, Validation Score = 0.9511
n_estimators = 100, Validation Score = 0.9511
n_estimators = 200, Validation Score = 0.9511
n_estimators = 300, Validation Score = 0.9511


In [22]:
# Final model using the selected hyperparameter
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train the final model on the training set
final_model.fit(X_train, y_train)

# Evaluate on the test set ONCE
test_score = final_model.score(X_test, y_test)

print(f"Final Test Score: {test_score:.4f}")

Final Test Score: 0.9462


## Final Test Evaluation

After tuning the `n_estimators` hyperparameter using the validation set, the final Random Forest model was evaluated on the test set exactly once.

The model achieved a validation accuracy of 95.11% and a final test accuracy of 94.62%.

The test score is slightly lower than the validation score, which is expected because the test set contains unseen data. The small difference suggests that the model's performance is relatively consistent on unseen data.